In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

# ==================== CONFIGURATION ====================
CSV_FILE = 'radio_signatures_large.csv'

THRESHOLD_FREQ = 0.5      # MHz
THRESHOLD_SIGNAL = 10     # dB

# Configuration pour éviter les crashs de kernel
plt.switch_backend('Agg')   # Utilise backend non-interactif
# ======================================================

print("Chargement du fichier...")

# Vérification du fichier
if not os.path.exists(CSV_FILE):
    print(f"❌ Erreur : Le fichier {CSV_FILE} n'existe pas !")
    print("Vérifiez le chemin du fichier.")
else:
    # Chargement des données
    df = pd.read_csv(CSV_FILE)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp')

    MMSI=np.random.choice(df['mmsi'])
    # Filtrage
    ship_df = df[df['mmsi'] == MMSI].copy()

    if ship_df.empty:
        print(f"❌ Aucune donnée trouvée pour le MMSI {MMSI}")
    else:
        print(f"✅ {len(ship_df)} enregistrements trouvés pour MMSI {MMSI}")

        # Détection des changements
        ship_df['freq_diff'] = ship_df['frequency'].diff()
        ship_df['signal_diff'] = ship_df['signal_strength'].diff()

        changements_freq = ship_df[abs(ship_df['freq_diff']) > THRESHOLD_FREQ]
        changements_signal = ship_df[abs(ship_df['signal_diff']) > THRESHOLD_SIGNAL]

        # Affichage résultats
        print("\n=== Changements brutaux de Fréquence (> {:.1f} MHz) ===".format(THRESHOLD_FREQ))
        if not changements_freq.empty:
            print(changements_freq[['timestamp', 'frequency', 'freq_diff']].round(3))
        else:
            print("Aucun changement brutal détecté.")

        print("\n=== Variations brutales de Signal Strength (> {:.0f} dB) ===".format(THRESHOLD_SIGNAL))
        if not changements_signal.empty:
            print(changements_signal[['timestamp', 'signal_strength', 'signal_diff']].round(2))
        else:
            print("Aucune variation brutale détectée.")

        # ==================== GRAPHIQUE ====================
        fig, ax1 = plt.subplots(figsize=(14, 7))

        ax1.plot(ship_df['timestamp'], ship_df['frequency'], 'b-', linewidth=2, label='Frequency (MHz)')
        ax1.set_xlabel('Temps')
        ax1.set_ylabel('Fréquence (MHz)', color='blue')
        ax1.tick_params(axis='y', labelcolor='blue')

        ax2 = ax1.twinx()
        ax2.plot(ship_df['timestamp'], ship_df['signal_strength'], 'r-', linewidth=2, label='Signal Strength (dBm)')
        ax2.set_ylabel('Signal Strength (dBm)', color='red')
        ax2.tick_params(axis='y', labelcolor='red')

        plt.title(f'Évolution Frequency & Signal Strength - MMSI {MMSI}')
        fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.9))
        plt.grid(True, alpha=0.3)

        # Sauvegarde au lieu de plt.show() (plus stable)
        output_file = f'evolution_mmsi_{MMSI}.png'
        plt.savefig(output_file, dpi=300, bbox_inches='tight')
        plt.close()   # Important pour libérer la mémoire

        print(f"\n✅ Graphique sauvegardé sous : {output_file}")

Chargement du fichier...
✅ 7 enregistrements trouvés pour MMSI 645475822

=== Changements brutaux de Fréquence (> 0.5 MHz) ===
                     timestamp  frequency  freq_diff
4004 2026-01-30 00:00:00+00:00     161.92       4.59
1886 2026-02-11 02:00:00+00:00     161.16      -0.76
2856 2026-04-21 04:00:00+00:00     159.77      -1.39
1212 2026-05-10 23:00:00+00:00     160.84       1.07
124  2026-09-26 06:00:00+00:00     159.59      -1.25
4998 2026-10-22 00:00:00+00:00     158.69      -0.90

=== Variations brutales de Signal Strength (> 10 dB) ===
                     timestamp  signal_strength  signal_diff
4004 2026-01-30 00:00:00+00:00            -98.2        -57.3
1886 2026-02-11 02:00:00+00:00            -44.8         53.4
2856 2026-04-21 04:00:00+00:00           -106.0        -61.2
1212 2026-05-10 23:00:00+00:00            -27.2         78.8
124  2026-09-26 06:00:00+00:00           -106.3        -79.1

✅ Graphique sauvegardé sous : evolution_mmsi_645475822.png


In [3]:
df[df['mmsi'] == np.random.choice(df['mmsi'])]

,signature_id,mmsi,frequency,bandwidth,modulation,power,timestamp,location_lat,location_lon,signal_strength,noise_level,pulse_pattern,signal_to_noise_ratio
762,SIG-00762,458498610,158.59,41.0,OFDM,23.2,2026-03-12 02:00:00+00:00,13.5340,-50.0889,-37.6,-86.9,Long-Short-Long,13.9
2819,SIG-02819,458498610,157.01,43.6,DSC,461.0,2026-04-06 20:00:00+00:00,-47.9483,-89.0409,-57.3,-61.2,Continuous,20.0
3800,SIG-03800,458498610,161.96,31.1,DSC,4.1,2026-06-29 04:00:00+00:00,37.1583,-167.1003,-23.4,-78.5,Short-Short-Long,15.7
365,SIG-00365,458498610,161.04,22.4,AM,58.1,2026-07-15 19:00:00+00:00,7.8605,87.7670,-111.9,-67.4,Continuous,47.0
2437,SIG-02437,458498610,161.97,20.6,AM,123.2,2026-12-03 23:00:00+00:00,-42.6962,83.1827,-70.7,-68.7,Short-Short-Short,9.7


In [6]:
ships_large=pd.read_csv('ships_large.csv')

In [8]:
merged=pd.merge(df, ships_large, on='mmsi', how='left')
merged=merged[["mmsi",'frequency','flag']]

In [10]:
# Vérification de la colonne 'flag'
if 'flag' not in merged.columns:
    print("❌ Erreur : La colonne 'flag' n'existe pas dans le fichier.")
    print("Colonnes disponibles :", merged.columns.tolist())
else:
    # Conversion timestamp (au cas où)
    if 'timestamp' in merged.columns:
        merged['timestamp'] = pd.to_datetime(merged['timestamp'])

    # Calcul de la moyenne et écart-type par pavillon
    stats = merged.groupby('flag')['frequency'].agg([
        ('Moyenne_Frequency', 'mean'),
        ('Ecart_Type', 'std'),
        ('Nombre_Signatures', 'count')
    ]).round(4)

    # Tri par moyenne décroissante
    stats = stats.sort_values('Moyenne_Frequency', ascending=False)

    print("\n" + "="*60)
    print("STATISTIQUES DE FRÉQUENCE PAR PAVILLON")
    print("="*60)
    print(stats)

    # Pavillon avec la fréquence moyenne la plus élevée
    pavillon_max = stats.index[0]
    freq_max = stats.iloc[0]['Moyenne_Frequency']

    print("\n" + "="*60)
    print(f"🚩 PAVILLON AVEC LA FRÉQUENCE MOYENNE LA PLUS ÉLEVÉE :")
    print(f"   {pavillon_max} → {freq_max:.4f} MHz")
    print("="*60)

    # Top 5 des pavillons
    print("\nTop 5 des pavillons par fréquence moyenne :")
    print(stats.head(5)[['Moyenne_Frequency', 'Ecart_Type', 'Nombre_Signatures']])


STATISTIQUES DE FRÉQUENCE PAR PAVILLON
                  Moyenne_Frequency  Ecart_Type  Nombre_Signatures
flag                                                              
Denmark                    159.1652      1.7734                478
Bahamas                    159.0598      1.7307                377
Panama                     159.0445      1.7571                640
Marshall Islands           159.0169      1.7181                440
Liberia                    158.9776      1.7927                514
Malta                      158.9753      1.7268                590
USA                        158.9724      1.7450                439
China                      158.9580      1.7374                519
Singapore                  158.9345      1.6932                517
France                     158.8932      1.7485                486

🚩 PAVILLON AVEC LA FRÉQUENCE MOYENNE LA PLUS ÉLEVÉE :
   Denmark → 159.1652 MHz

Top 5 des pavillons par fréquence moyenne :
                  Moyenne_Freq

In [14]:
ais_data_large=pd.read_csv('ais_data_large.csv')
merged2=pd.merge(df, ais_data_large, on='mmsi', how='left')
merged2=merged2[["mmsi",'frequency','speed']]

In [15]:
import pandas as pd
import scipy.stats as stats
import numpy as np

# Vérification des colonnes nécessaires
required_cols = ['speed', 'frequency']
missing = [col for col in required_cols if col not in merged2.columns]

if missing:
    print(f"❌ Colonnes manquantes : {missing}")
    print("Colonnes disponibles :", merged2.columns.tolist())
else:
    print(f"✅ Analyse sur {len(merged2)} enregistrements")

    # Nettoyage des données (suppression des NaN)
    df_clean = merged2[['speed', 'frequency']].dropna()

    if len(df_clean) < 2:
        print("❌ Pas assez de données valides pour calculer la corrélation.")
    else:
        # Calcul de la corrélation de Pearson
        corr_coeff, p_value = stats.pearsonr(df_clean['speed'], df_clean['frequency'])

        print("\n" + "="*55)
        print("CORRÉLATION ENTRE SPEED (AIS) ET FREQUENCY (Radio)")
        print("="*55)
        print(f"Coefficient de corrélation de Pearson : {corr_coeff:.4f}")
        print(f"p-value                                : {p_value:.6f}")
        print(f"Nombre de paires valides               : {len(df_clean)}")

        # Interprétation
        if abs(corr_coeff) < 0.1:
            strength = "très faible"
        elif abs(corr_coeff) < 0.3:
            strength = "faible"
        elif abs(corr_coeff) < 0.5:
            strength = "modérée"
        else:
            strength = "forte"

        direction = "positive" if corr_coeff > 0 else "négative"

        print(f"\nInterprétation : Corrélation {strength} {direction}")

        if p_value < 0.05:
            print("✅ Il y a une corrélation **statistiquement significative** (p < 0.05)")
        else:
            print("❌ La corrélation n'est **pas statistiquement significative** (p >= 0.05)")

        # Corrélation par pavillon (optionnel mais utile)
        if 'flag' in merged2.columns:
            print("\n" + "-"*40)
            print("Corrélation par pavillon :")
            for flag, group in merged2.groupby('flag'):
                clean_group = group[['speed', 'frequency']].dropna()
                if len(clean_group) > 10:  # minimum pour fiabilité
                    r, p = stats.pearsonr(clean_group['speed'], clean_group['frequency'])
                    print(f"  {flag:15} : r = {r:.4f}, p = {p:.6f}")

✅ Analyse sur 49858 enregistrements

CORRÉLATION ENTRE SPEED (AIS) ET FREQUENCY (Radio)
Coefficient de corrélation de Pearson : 0.0036
p-value                                : 0.416554
Nombre de paires valides               : 49858

Interprétation : Corrélation très faible positive
❌ La corrélation n'est **pas statistiquement significative** (p >= 0.05)
